## Adaptive Spectral Modulation Block 可解释性分析

## 从训练好的 checkpoint 中读取 `ASM_all` 的频域复数权重：
- 参数路径：`encoder.encoder_layers.{L}.asmModule.frequency_weight_global / frequency_weight_local`
- 原始形状：`(F, C, 2)`，先经 `torch.view_as_complex` 得到 `(F, C)` 复数权重再可视化
- 折线图：对各层 frequency_weight 的通道均值、freq_mask 通过率和 gate 均值按频率作图

In [2]:
import os
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.ticker import FuncFormatter
import numpy as np
import torch

# 将项目根目录加入 sys.path
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'run.py').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [3]:
# ========== 路径与可视化配置 ==========
save_path = PROJECT_ROOT / 'figs'
ASM_OUTPUT_DIR = save_path / 'Adaptive Spectral Modulation'
ASM_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 采样率（Hz），用于 x 轴频率刻度；CAEEG 1.0s 窗口默认 128 Hz
SAMPLING_RATE = 128

# 横坐标固定显示的频率刻度（Hz）
FREQ_TICK_HZ = [4, 8, 13, 30, 45]

# 各层折线图横轴右边界 (Hz)，layer_idx 0~5 对应第 1~6 层
LAYER_FREQ_XMAX = {
    0: 64,
    1: 32,
    2: 16,
    3: 8,
    4: 4,
    5: 2,
}

# 复数权重可视化分量：'real' | 'imag' | 'abs'（模长）
WEIGHT_COMPONENT = 'abs'
# 绘图样式
font_path = PROJECT_ROOT / 'figs/fonts/TIMES.TTF'
if font_path.is_file():
    font_manager.fontManager.addfont(font_path)
PLOT_FONT = 'Times New Roman'

# ---------- 折线图样式（修改下列参数即可统一调整所有折线图） ----------
LINE_FONT_XTICK = 12        # 横坐标刻度字号
LINE_FONT_YTICK = 12        # 纵坐标刻度字号
LINE_FONT_XLABEL = 15      # 横坐标标题字号
LINE_FONT_YLABEL = 15      # 纵坐标标题字号
LINE_FONT_LEGEND = 10       # 图例字号
LINE_FONT_TITLE = 10       # 图标题字号
LINE_WIDTH = 1.8           # 曲线线宽
LINE_GRID_X_WIDTH = 2.0    # X 轴网格线宽
LINE_GRID_Y_WIDTH = 1.0    # Y 轴网格线宽
LINE_YTICK_DECIMAL = 1     # 纵坐标刻度小数位数（配合 ×10^n 偏移显示尾数）
LINE_YSCALE_LABEL_X = 0.07  # ×10^n 横向位置（axes 坐标，0=左边界，1=右边界）
LINE_YSCALE_LABEL_Y = 0.995   # ×10^n 纵向位置（axes 坐标，0=下边界，1=上边界）

LINE_FIGSIZE = (6.0, 1.8)
LINE_FIG_DPI = 600
# 画布边距（figure 坐标 0~1）；减小边距可放大中心绘图区
LINE_FIG_MARGINS = dict(left=0.1, bottom=0.13, right=0.99, top=0.92)
plt.rcParams.update({'font.family': PLOT_FONT, 'font.serif': [PLOT_FONT]})

print(f'PROJECT_ROOT   = {PROJECT_ROOT}')
print(f'ASM_OUTPUT_DIR = {ASM_OUTPUT_DIR}')
print(f'WEIGHT_COMPONENT = {WEIGHT_COMPONENT}')

PROJECT_ROOT   = /root/autodl-tmp/ADSST-Net
ASM_OUTPUT_DIR = /root/autodl-tmp/ADSST-Net/figs/Adaptive Spectral Modulation
WEIGHT_COMPONENT = abs


In [4]:
# ========== checkpoint 路径配置 ==========
PROJECT_NAME = 'CAEEG3_1s_00_EO_Aug-ADSSTNet-NCvsMCIvsAD-oneFreBanDAGnoRanCrop'
CHECKPOINT_ROOT = PROJECT_ROOT / 'checkpoints' / PROJECT_NAME

fold_dirs = sorted([d for d in CHECKPOINT_ROOT.iterdir() if d.is_dir()]) if CHECKPOINT_ROOT.is_dir() else []
print(f'CHECKPOINT_ROOT = {CHECKPOINT_ROOT}')
print(f'发现折目录 {len(fold_dirs)} 个: {[d.name for d in fold_dirs]}')

CHECKPOINT_ROOT = /root/autodl-tmp/ADSST-Net/checkpoints/CAEEG3_1s_00_EO_Aug-ADSSTNet-NCvsMCIvsAD-oneFreBanDAGnoRanCrop
发现折目录 5 个: ['itr_000', 'itr_001', 'itr_002', 'itr_003', 'itr_004']


### 工具函数：读取 checkpoint 中的频域权重

In [5]:
# 与训练脚本 ADSSTNet-Ablation-Experiment.sh 一致
INPUT_SEQ_LEN = 128
DOWNSAMPLE_MIN_LEN = 4


def _normalize_state_key(key: str) -> str:
    return key.replace('module.', '')


def compute_seq_each_layer(seq_len: int, downsample_min_len: int | None) -> list[int]:
    """复现 Encoder 中 seq_each_layer 的计算逻辑（层间 AvgPool(2) 折半）。"""
    if downsample_min_len is None or downsample_min_len >= seq_len:
        return [seq_len]
    seq_list = [seq_len]
    L = seq_len
    while L > downsample_min_len:
        L = max(1, (L - 2) // 2 + 1)
        seq_list.append(L)
    return seq_list


def freq_axis_hz(n_freq: int, layer_idx: int, input_seq_len: int, fs: float) -> np.ndarray:
    """rfft 频率 bin 对应的物理频率（Hz）；每层有效采样率折半。"""
    seq_len = input_seq_len // (2 ** layer_idx)
    fs_eff = fs / (2 ** layer_idx)
    return np.arange(n_freq) * fs_eff / seq_len


def list_asm_layers(state_dict: dict) -> list[int]:
    """列出 checkpoint 中含 asmModule.frequency_weight_global 的 encoder 层索引。"""
    layers = set()
    pat = re.compile(r'^encoder\.encoder_layers\.(\d+)\.asmModule\.frequency_weight_global$')
    for key in state_dict:
        m = pat.match(_normalize_state_key(key))
        if m:
            layers.add(int(m.group(1)))
    return sorted(layers)


def load_checkpoint_state(fold_dir: Path) -> dict:
    ckpt_path = fold_dir / 'checkpoint.pth'
    if not ckpt_path.is_file():
        raise FileNotFoundError(f'未找到 checkpoint: {ckpt_path}')
    ckpt = torch.load(ckpt_path, map_location='cpu')
    if isinstance(ckpt, dict) and 'state_dict' in ckpt:
        return ckpt['state_dict']
    if isinstance(ckpt, dict) and 'model' in ckpt:
        return ckpt['model']
    return ckpt


def extract_frequency_weight(
    state_dict: dict,
    layer_idx: int,
    weight_type: str = 'global',
    component: str = WEIGHT_COMPONENT,
) -> tuple[np.ndarray, int, int]:
    """
    从 state_dict 提取 frequency_weight_{global|local}，经 view_as_complex 后返回绘图矩阵。

    Returns
    -------
    mat : (C, F) float ndarray，行=通道、列=频率
    n_freq, n_ch : 频率 bin 数、通道数
    """
    suffix = f'frequency_weight_{weight_type}'
    key_pat = re.compile(rf'^encoder\.encoder_layers\.{layer_idx}\.asmModule\.{suffix}$')
    param_key = None
    for key in state_dict:
        if key_pat.match(_normalize_state_key(key)):
            param_key = key
            break
    if param_key is None:
        avail = list_asm_layers(state_dict)
        raise KeyError(
            f'layer {layer_idx} 未找到 asmModule.{suffix}；checkpoint 可用层: {avail}'
        )

    raw = state_dict[param_key]  # (F, C, 2)
    weight_complex = torch.view_as_complex(raw)  # (F, C)

    if component == 'real':
        values = weight_complex.real
    elif component == 'imag':
        values = weight_complex.imag
    elif component == 'abs':
        values = weight_complex.abs()
    else:
        raise ValueError(f"component 应为 'real' | 'imag' | 'abs'，收到: {component}")

    mat = values.detach().cpu().numpy().T  # (C, F)
    n_ch, n_freq = mat.shape
    return mat, n_freq, n_ch


def _pick_freq_tick_indices(freqs: np.ndarray, tick_hz: list | None = None) -> tuple[np.ndarray, list[str]]:
    """将固定频率刻度映射到最近的 rfft bin；超出当前层 Nyquist 的刻度不显示。"""
    if tick_hz is None:
        tick_hz = FREQ_TICK_HZ
    max_hz = float(freqs[-1])
    indices, labels = [], []
    for hz in tick_hz:
        if hz > max_hz + 1e-6:
            continue
        idx = int(np.argmin(np.abs(freqs - hz)))
        if indices and idx == indices[-1]:
            continue
        indices.append(idx)
        labels.append(f'{int(hz)}' if float(hz).is_integer() else f'{hz}')
    return np.asarray(indices, dtype=int), labels


def _pick_freq_tick_values(x_max: float, tick_hz: list | None = None) -> tuple[list[float], list[str]]:
    """按层统一横轴范围选取刻度：优先 FREQ_TICK_HZ，低层无匹配时用整数 Hz。"""
    if tick_hz is None:
        tick_hz = FREQ_TICK_HZ
    ticks, labels = [], []
    for hz in tick_hz:
        if hz > x_max + 1e-6:
            continue
        ticks.append(float(hz))
        labels.append(f'{int(hz)}' if float(hz).is_integer() else f'{hz}')
    if not ticks and x_max >= 1:
        ticks = [float(i) for i in range(int(x_max) + 1)]
        labels = [str(i) for i in range(int(x_max) + 1)]
    elif not ticks:
        ticks, labels = [0.0, x_max], ['0', f'{x_max:g}']
    return ticks, labels


def _layer_freq_xmax(layer_idx: int) -> float:
    """各层折线图横轴右边界 (Hz)。"""
    if layer_idx not in LAYER_FREQ_XMAX:
        raise KeyError(f'layer_idx={layer_idx} 未在 LAYER_FREQ_XMAX 中配置')
    return float(LAYER_FREQ_XMAX[layer_idx])


def _create_line_fig_ax():
    fig, ax = plt.subplots(figsize=LINE_FIGSIZE, facecolor='white')
    fig.patch.set_facecolor('white')
    ax.set_facecolor('white')
    fig.subplots_adjust(**LINE_FIG_MARGINS)
    return fig, ax


def _style_line_fig_ax(ax):
    ax.grid(True, axis='y', color='gray', alpha=0.3, linestyle='--', linewidth=LINE_GRID_Y_WIDTH)
    ax.grid(True, axis='x', color='gray', alpha=0.5, linewidth=LINE_GRID_X_WIDTH, linestyle='--')
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_edgecolor('black')
        spine.set_linewidth(1.0)


def _set_freq_xaxis(ax, freqs: np.ndarray, layer_idx: int | None = None):
    if layer_idx is not None:
        x_max = _layer_freq_xmax(layer_idx)
        tick_vals, tick_labels = _pick_freq_tick_values(x_max)
        ax.set_xticks(tick_vals)
    else:
        x_max = float(freqs[-1])
        tick_idx, tick_labels = _pick_freq_tick_indices(freqs)
        ax.set_xticks(freqs[tick_idx])
    ax.set_xticklabels(tick_labels, fontsize=LINE_FONT_XTICK, fontfamily=PLOT_FONT)
    ax.set_xlabel('Frequency (Hz)', fontsize=LINE_FONT_XLABEL, fontfamily=PLOT_FONT)
    ax.set_xlim(left=0, right=x_max)


def _set_line_ylabel(ax, label: str):
    ax.set_ylabel(label, fontsize=LINE_FONT_YLABEL, fontfamily=PLOT_FONT)


def _y_axis_scale_exp(max_abs: float) -> int:
    """根据纵轴量级返回 ×10^n 的指数 n。"""
    if max_abs < 1e-15:
        return 0
    return int(np.floor(np.log10(max_abs)))


def _format_times10_exp(exp: int) -> str:
    """将指数转为 ×10^n 文本（Unicode 上标，便于使用 Times New Roman）。"""
    sup = '⁰¹²³⁴⁵⁶⁷⁸⁹'
    if exp == 0:
        return ''
    if exp < 0:
        body = ''.join(sup[int(c)] for c in str(-exp))
        return f'×10⁻{body}'
    body = ''.join(sup[int(c)] for c in str(exp))
    return f'×10{body}'


def _style_line_yticks(ax):
    """纵轴刻度：尾数保留 LINE_YTICK_DECIMAL 位小数；必要时在图内绘制 ×10^n。"""
    ymin, ymax = ax.get_ylim()
    max_abs = max(abs(ymin), abs(ymax))
    exp = _y_axis_scale_exp(max_abs)
    scale = 10.0 ** exp
    dec = LINE_YTICK_DECIMAL

    def _fmt(y, _pos):
        if abs(y) < 1e-15:
            return f'{0:.{dec}f}'
        return f'{y / scale:.{dec}f}'

    ax.yaxis.set_major_formatter(FuncFormatter(_fmt))
    ax.yaxis.get_offset_text().set_visible(False)

    old_label = getattr(ax, '_yscale_label_artist', None)
    if old_label is not None:
        old_label.remove()
        ax._yscale_label_artist = None

    if exp != 0:
        yscale_fp = font_manager.FontProperties(
            family=PLOT_FONT, size=LINE_FONT_YTICK,
        )
        ax._yscale_label_artist = ax.text(
            LINE_YSCALE_LABEL_X, LINE_YSCALE_LABEL_Y,
            _format_times10_exp(exp),
            transform=ax.transAxes,
            ha='right', va='bottom',
            fontproperties=yscale_fp,
            clip_on=False,
        )

    for tick_label in ax.get_yticklabels():
        tick_label.set_fontfamily(PLOT_FONT)
        tick_label.set_fontsize(LINE_FONT_YTICK)


def _style_line_legend(ax, loc='best'):
    ax.legend(
        loc=loc,
        prop={'family': PLOT_FONT, 'size': LINE_FONT_LEGEND},
        facecolor='white',
        framealpha=1.0,
    )


def _set_line_title(ax, title: str):
    ax.set_title(
        title,
        fontsize=LINE_FONT_TITLE, fontweight='bold', fontfamily=PLOT_FONT,
    )


def _save_line_fig(fig, save_path: Path):
    save_path = Path(save_path)
    if save_path.suffix.lower() != '.svg':
        save_path = save_path.with_suffix('.svg')
    fig.savefig(save_path, format='svg', bbox_inches='tight', facecolor='white')
    plt.close(fig)


SEQ_EACH_LAYER = compute_seq_each_layer(INPUT_SEQ_LEN, DOWNSAMPLE_MIN_LEN)
print(f'INPUT_SEQ_LEN={INPUT_SEQ_LEN}, DOWNSAMPLE_MIN_LEN={DOWNSAMPLE_MIN_LEN}')
print(f'各层 seq_len: {SEQ_EACH_LAYER}')
print(f'各层频率 bin 数 F: {[L // 2 + 1 for L in SEQ_EACH_LAYER]}')

INPUT_SEQ_LEN=128, DOWNSAMPLE_MIN_LEN=4
各层 seq_len: [128, 64, 32, 16, 8, 4]
各层频率 bin 数 F: [65, 33, 17, 9, 5, 3]


### 绘制 itr_00* 各层 frequency_weight 折线图

每一层单独保存一幅图，`frequency_weight_global` 与 `frequency_weight_local` 绘制在同一张图中对比。

In [6]:
def plot_frequency_weight_mean_layer(
    mean_global: np.ndarray,
    mean_local: np.ndarray,
    layer_idx: int,
    fold_tag: str,
    save_path: Path,
    input_seq_len: int = INPUT_SEQ_LEN,
    fs: float = SAMPLING_RATE,
):
    """单层 frequency_weight 通道均值折线图：global 与 local 同图对比。"""
    n_freq = len(mean_global)
    freqs = freq_axis_hz(n_freq, layer_idx, input_seq_len, fs)
    comp_label = {'real': 'Re', 'imag': 'Im', 'abs': '|w|'}[WEIGHT_COMPONENT]

    fig, ax = _create_line_fig_ax()
    ax.plot(
        freqs, mean_global, linewidth=LINE_WIDTH, color='#1f77b4',
        label='$W_G$',
    )
    ax.plot(
        freqs, mean_local, linewidth=LINE_WIDTH, color='#ff7f0e',
        label='$W_L$',
    )
    _set_freq_xaxis(ax, freqs, layer_idx=layer_idx)
    _set_line_ylabel(ax, 'Weight')
    if WEIGHT_COMPONENT == 'abs':
        ax.set_ylim(bottom=0.0)
    _style_line_fig_ax(ax)
    _style_line_yticks(ax)
    _style_line_legend(ax)
    # seq_len_layer = input_seq_len // (2 ** layer_idx)
    # fs_eff = fs / (2 ** layer_idx)
    # _set_line_title(
    #     ax,
    #     f'{fold_tag} | Layer {layer_idx} | seq_len={seq_len_layer}, fs={fs_eff:.0f} Hz, F={n_freq}',
    # )
    _save_line_fig(fig, save_path)


# ========== 绘制 itr_00* 各层 frequency_weight 通道均值折线图 ==========
FOLD_TAGS = sorted(
    d.name for d in CHECKPOINT_ROOT.iterdir()
    if d.is_dir() and re.match(r'^itr_00', d.name)
)
if not FOLD_TAGS:
    raise FileNotFoundError(f'CHECKPOINT_ROOT 下未找到 itr_00* 折目录: {CHECKPOINT_ROOT}')

saved_paths = []
for fold_tag in FOLD_TAGS:
    if fold_tag != "itr_001": continue
    fold_dir = CHECKPOINT_ROOT / fold_tag
    state = load_checkpoint_state(fold_dir)
    layer_ids = list_asm_layers(state)
    if not layer_ids:
        print(f'[skip] {fold_tag}: 未找到 asmModule.frequency_weight_global')
        continue

    fold_out_dir = ASM_OUTPUT_DIR / fold_tag
    fold_out_dir.mkdir(parents=True, exist_ok=True)

    for layer_idx in layer_ids:
        mat_global, n_freq, n_ch = extract_frequency_weight(
            state, layer_idx, weight_type='global',
        )
        mat_local, _, _ = extract_frequency_weight(
            state, layer_idx, weight_type='local',
        )
        mean_global = mat_global.mean(axis=0)  # (F,) 各频率下所有通道权重的均值
        mean_local = mat_local.mean(axis=0)

        fname = (
            f'{fold_tag}_layer{layer_idx:02d}_frequency_weight_'
            f'{WEIGHT_COMPONENT}_mean_line.png'
        )
        out_path = fold_out_dir / fname
        plot_frequency_weight_mean_layer(
            mean_global, mean_local, layer_idx, fold_tag, out_path,
            input_seq_len=INPUT_SEQ_LEN, fs=SAMPLING_RATE,
        )
        saved_paths.append(out_path)
        print(
            f'已保存: {out_path.name}  '
            f'mean_shape=({n_freq},), channels={n_ch}'
        )

print(f'\n共保存 {len(saved_paths)} 张折线图 -> {ASM_OUTPUT_DIR}')



已保存: itr_001_layer00_frequency_weight_abs_mean_line.png  mean_shape=(65,), channels=20
已保存: itr_001_layer01_frequency_weight_abs_mean_line.png  mean_shape=(33,), channels=20
已保存: itr_001_layer02_frequency_weight_abs_mean_line.png  mean_shape=(17,), channels=20
已保存: itr_001_layer03_frequency_weight_abs_mean_line.png  mean_shape=(9,), channels=20
已保存: itr_001_layer04_frequency_weight_abs_mean_line.png  mean_shape=(5,), channels=20
已保存: itr_001_layer05_frequency_weight_abs_mean_line.png  mean_shape=(3,), channels=20

共保存 6 张折线图 -> /root/autodl-tmp/ADSST-Net/figs/Adaptive Spectral Modulation


## 计算模型输出的 freq_mask 和 gate（按人群统计）

从测试集前向推理收集各 encoder 层的 `freq_mask` (B, F, C) 与 `gate` (B, F)：
- **freq_mask**：按 NC / MCI / AD 分组，统计每个通道上各频率被掩码的样本次数，得到 (C, F) 计数矩阵
- **gate**：按人群分组求均值，得到 (F,) 平均门控曲线

可选择折（`FOLD_IDX`），每层单独保存统计结果与可视化图。

In [11]:
import random
from argparse import Namespace

import torch
from torch.optim.swa_utils import AveragedModel
from torch.utils.data import DataLoader

from data_provider.data_factory import data_provider
from data_provider.uea import collate_fn
from models import ADSSTNet
from utils.tools import compute_pearson_A0_subject_mean

# ========== freq_mask / gate 分析配置 ==========
MASK_GATE_OUTPUT_DIR = ASM_OUTPUT_DIR
MASK_GATE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BASE_SEED = 38
ITRS = 5
FOLD_IDX = 1                   # 可选 0..4，对应 itr_000..itr_004
USE_SWA = True
CLASS_NAMES = ['HC', 'MCI', 'AD']
CLASS_IDS = [0, 1, 2]

COLLECT_CORRECT_ONLY = True    # True 时仅保留分类正确样本
BALANCE_CLASSES = True          # 按各类最少样本数均衡

MODEL_ARGS_BASE = dict(
    task_name='supervised',
    model='ADSSTNet',
    data='CAEEG',
    root_path=str((PROJECT_ROOT / 'dataset/CAEEG3_1.0s_0.0_EO_Aug').resolve()) + '/',
    selected_classes=[0, 1, 2],
    cross_val='mccv',
    use_augmentation=True,
    e_layers=0,
    d_model=20,
    expand=2,
    d_state=16,
    dt_rank=2,
    d_conv_in=4,
    d_conv=4,
    d_conv_out=4,
    bimamba=True,
    space=True,
    se_reduction=16,
    dim_UVA=4,
    topk_A0=4,
    embedding_dim=32,
    sample_adapt_A0=True,
    sample_adapt_Fre=True,
    projection_dim=128,
    dropout_head=0.7,
    num_heads=4,
    downsample_min_len=4,
    group_size=16,
    batch_size=256,
    num_workers=0,
    preserve_frontal_occipital=False,
)

A0_KWARGS = dict(
    use_abs=True,
    topk=MODEL_ARGS_BASE['topk_A0'],
    sym_mode='max',
    eps=1e-8,
    row_normalize=True,
    clip_negative=True,
    dataset_name=MODEL_ARGS_BASE['data'],
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'MASK_GATE_OUTPUT_DIR = {MASK_GATE_OUTPUT_DIR}')
print(f'DEVICE = {DEVICE}, USE_SWA = {USE_SWA}, FOLD_IDX = {FOLD_IDX}')

MASK_GATE_OUTPUT_DIR = /root/autodl-tmp/ADSST-Net/figs/Adaptive Spectral Modulation
DEVICE = cuda, USE_SWA = True, FOLD_IDX = 1


### 模型初始化、测试集加载与数据收集

In [12]:
def _as_layer_list(x):
    if isinstance(x, (list, tuple)):
        return list(x)
    return [x]


ASM_KEYS = ('ASM_before', 'freq_mask', 'gate', 'ASM_after')


def _get_data(flag: str, fold_seed: int):
    fold_args = Namespace(**MODEL_ARGS_BASE, seed=fold_seed)
    random.seed(fold_seed)
    np.random.seed(fold_seed)
    return data_provider(fold_args, flag, log=None)


def build_adsstnet_for_fold(fold_seed: int):
    fold_args = Namespace(**MODEL_ARGS_BASE, seed=fold_seed)
    random.seed(fold_seed)
    np.random.seed(fold_seed)

    train_data, _ = data_provider(fold_args, 'TRAIN', log=None)
    fold_args.seq_len = train_data.max_seq_len
    fold_args.enc_in = train_data.X.shape[-1]
    fold_args.num_class = len(np.unique(train_data.y[:, 0]))
    fold_args.A0 = compute_pearson_A0_subject_mean(
        train_data, enc_in=fold_args.enc_in, **A0_KWARGS,
    )

    model = ADSSTNet.Model(fold_args).float().to(DEVICE)
    if USE_SWA:
        model = AveragedModel(model)
    model.eval()
    return model, fold_args


def load_test_loader(fold_args):
    test_data, _ = _get_data('TEST', fold_args.seed)
    test_loader = DataLoader(
        test_data,
        batch_size=fold_args.batch_size,
        shuffle=False,
        num_workers=fold_args.num_workers,
        drop_last=False,
        collate_fn=lambda x: collate_fn(x, max_len=fold_args.seq_len),
    )
    return test_data, test_loader


def load_fold_checkpoint(model, fold_dir: Path):
    state = torch.load(fold_dir / 'checkpoint.pth', map_location=DEVICE)
    model.load_state_dict(state)
    model.eval()


def _balance_class_indices(labels, class_ids, rng: np.random.Generator):
    labels = np.asarray(labels)
    class_ids = sorted(class_ids)
    per_class_idx = {c: np.where(labels == c)[0] for c in class_ids}
    counts = {c: len(per_class_idx[c]) for c in class_ids}
    missing = [c for c in class_ids if counts[c] == 0]
    if missing:
        raise RuntimeError(f'类别 {missing} 无样本，无法均衡采样')

    min_n = min(counts.values())
    selected = []
    for c in class_ids:
        idx = per_class_idx[c]
        chosen = idx if len(idx) == min_n else rng.choice(idx, size=min_n, replace=False)
        selected.extend(int(i) for i in chosen)
    rng.shuffle(selected)
    return selected, min_n, counts


def _subset_asm_samples(data_by_layer, label_list, keep_indices):
    label_list = [label_list[i] for i in keep_indices]
    subset = {
        layer_idx: {
            key: [data_by_layer[layer_idx][key][i] for i in keep_indices]
            for key in ASM_KEYS
        }
        for layer_idx in data_by_layer
    }
    return subset, label_list


def _asm_tensor_to_numpy(tensor, key, sample_idx):
    if tensor is None:
        return None
    arr = tensor[sample_idx].detach().cpu().numpy()
    if key == 'gate':
        return np.squeeze(arr)
    return arr


@torch.no_grad()
def collect_test_freq_mask_gate(
    model,
    test_loader,
    correct_only: bool = True,
    balance_classes: bool = True,
    balance_seed=None,
    class_ids=None,
):
    """在测试集上收集全部 encoder 层的 ASM_visualization。

    Returns
    -------
    data_by_layer : dict[int, dict[str, list[np.ndarray]]]
        每层包含 ASM_KEYS 四个键，值为样本列表。
    label_list : list[int]
    layer_ids : list[int]
    """
    data_by_layer = None
    label_list = []
    n_total = n_kept = 0

    for batch_x, label_id, _ in test_loader:
        batch_x_ori = batch_x[:, 0, :, :].float().to(DEVICE)
        labels = label_id[:, 0].cpu().numpy().astype(int)
        n_samples = len(labels)
        n_total += n_samples

        _, logits, _, asm_vis_list, _, _ = model(
            batch_x_ori, None, None, None, model_state='test',
        )
        preds = logits.argmax(dim=-1).detach().cpu().numpy().astype(int)
        sample_indices = (
            [i for i in range(n_samples) if preds[i] == labels[i]]
            if correct_only else list(range(n_samples))
        )
        if not sample_indices:
            continue
        n_kept += len(sample_indices)

        asm_layers = _as_layer_list(asm_vis_list)
        n_layers = len(asm_layers)

        if data_by_layer is None:
            data_by_layer = {
                layer_idx: {k: [] for k in ASM_KEYS}
                for layer_idx in range(n_layers)
            }

        for layer_idx in range(n_layers):
            asm_dict = asm_layers[layer_idx]
            for key in ASM_KEYS:
                if key not in asm_dict:
                    raise KeyError(
                        f'Layer {layer_idx} ASM_visualization 缺少键 {key!r}，'
                        f'现有键: {list(asm_dict.keys())}'
                    )
                tensor = asm_dict[key]
                for i in sample_indices:
                    data_by_layer[layer_idx][key].append(
                        _asm_tensor_to_numpy(tensor, key, i)
                    )
        label_list.extend(int(labels[i]) for i in sample_indices)

    if data_by_layer is None:
        raise RuntimeError('未收集到任何样本')

    if correct_only:
        print(f'collect_test_freq_mask_gate: 保留正确分类样本 {n_kept}/{n_total}')

    if balance_classes:
        balance_class_ids = sorted(set(class_ids if class_ids is not None else label_list))
        rng = np.random.default_rng(balance_seed)
        keep_indices, min_n, raw_counts = _balance_class_indices(
            label_list, balance_class_ids, rng,
        )
        orig_labels = label_list
        data_by_layer, label_list = _subset_asm_samples(
            data_by_layer, orig_labels, keep_indices,
        )
        count_str = ', '.join(
            f'{CLASS_NAMES[c] if c < len(CLASS_NAMES) else c}: {raw_counts[c]}->{min_n}'
            for c in balance_class_ids
        )
        print(f'collect_test_freq_mask_gate: 类别均衡采样 -> 每类 n={min_n} ({count_str})')

    layer_ids = sorted(data_by_layer.keys())
    return data_by_layer, label_list, layer_ids

### 统计分析与人群对比可视化

In [13]:
def compute_freq_mask_count_by_class(masks, labels, class_ids):
    """统计各人群在每个通道上各频率被掩码的样本次数，返回 {class_id: (C, F)}。
    """
    labels = np.asarray(labels)
    stats, counts = {}, {}
    for c in class_ids:
        idx = np.where(labels == c)[0]
        class_masks = [masks[i] for i in idx if masks[i] is not None]
        if not class_masks:
            continue
        stacked = np.stack(class_masks, axis=0)          # (N, F, C)
        masked = stacked == 1                            # 0/1 硬掩码，0 表示被掩码
        stats[c] = masked.sum(axis=0).T.astype(np.int32) # (C, F)
        counts[c] = len(class_masks)
    return stats, counts


def compute_gate_mean_by_class(gates, labels, class_ids):
    """统计各人群 gate 均值，返回 {class_id: (F,)} 与样本数。"""
    labels = np.asarray(labels)
    stats, counts = {}, {}
    for c in class_ids:
        idx = np.where(labels == c)[0]
        class_gates = [gates[i] for i in idx if gates[i] is not None]
        if not class_gates:
            continue
        stats[c] = np.mean(np.stack(class_gates, axis=0), axis=0)
        counts[c] = len(class_gates)
    return stats, counts


def plot_gate_mean_layer(
    class_stats: dict,
    class_counts: dict,
    layer_idx: int,
    fold_tag: str,
    save_path: Path,
    input_seq_len: int = INPUT_SEQ_LEN,
    fs: float = SAMPLING_RATE,
):
    """单层 gate 人群均值曲线对比。"""
    class_ids = [c for c in CLASS_IDS if c in class_stats]
    if not class_ids:
        return
    n_freq = len(class_stats[class_ids[0]])
    freqs = freq_axis_hz(n_freq, layer_idx, input_seq_len, fs)

    fig, ax = _create_line_fig_ax()
    category_colors = {0: 'green', 1: 'orange', 2: 'red'}
    for c in class_ids:
        ax.plot(
            freqs, class_stats[c], linewidth=LINE_WIDTH,
            color=category_colors.get(c, 'black'),
            label=f"{CLASS_NAMES[c]}",
        )
    _set_freq_xaxis(ax, freqs, layer_idx=layer_idx)
    _set_line_ylabel(ax, r'$G_{f}$')
    ax.set_ylim(0.0, 1.0)
    _style_line_fig_ax(ax)
    _style_line_yticks(ax)
    _style_line_legend(ax)
    # seq_len_layer = input_seq_len // (2 ** layer_idx)
    # fs_eff = fs / (2 ** layer_idx)
    # _set_line_title(
    #     ax,
    #     f'{fold_tag} | Layer {layer_idx} | seq_len={seq_len_layer}, fs={fs_eff:.0f} Hz, F={n_freq}',
    # )
    _save_line_fig(fig, save_path)


def plot_freq_mask_pass_rate_layer(
    class_stats: dict,
    class_counts: dict,
    layer_idx: int,
    fold_tag: str,
    save_path: Path,
    input_seq_len: int = INPUT_SEQ_LEN,
    fs: float = SAMPLING_RATE,
):
    """单层 freq_mask 人群通过率曲线：各频率点上通道平均通过率（未掩码样本数 / 总样本数）。"""
    class_ids = [c for c in CLASS_IDS if c in class_stats]
    if not class_ids:
        return
    n_ch, n_freq = class_stats[class_ids[0]].shape
    freqs = freq_axis_hz(n_freq, layer_idx, input_seq_len, fs)

    fig, ax = _create_line_fig_ax()
    category_colors = {0: 'green', 1: 'orange', 2: 'red'}
    all_pass_rates = []
    for c in class_ids:
        n = class_counts[c]
        pass_rate_cf = class_stats[c] / n                      # (C, F)，未掩码比例
        pass_rate_mean = pass_rate_cf.mean(axis=0)             # (F,) 通道平均
        all_pass_rates.append(pass_rate_mean)
        ax.plot(
            freqs, pass_rate_mean, linewidth=LINE_WIDTH,
            color=category_colors.get(c, 'black'),
            label=f"{CLASS_NAMES[c]}",
        )
    y_min = min(float(np.min(v)) for v in all_pass_rates)
    y_max = max(float(np.max(v)) for v in all_pass_rates)
    y_range = y_max - y_min
    pad = max(y_range * 0.08, 0.02)
    if y_range < 1e-9:
        y_min, y_max = max(0.0, y_min - 0.02), y_max + 0.02
    ax.set_ylim(max(0.0, y_min - pad * 0.5), y_max + pad)
    _set_freq_xaxis(ax, freqs, layer_idx=layer_idx)
    _set_line_ylabel(ax, 'Pass Rate')
    _style_line_fig_ax(ax)
    _style_line_yticks(ax)
    _style_line_legend(ax)
    # seq_len_layer = input_seq_len // (2 ** layer_idx)
    # fs_eff = fs / (2 ** layer_idx)
    # _set_line_title(
    #     ax,
    #     f'{fold_tag} | Layer {layer_idx} | seq_len={seq_len_layer}, fs={fs_eff:.0f} Hz, F={n_freq}, C={n_ch}',
    # )
    _save_line_fig(fig, save_path)

### 选定折收集并统计 freq_mask / gate

In [21]:
FOLD_TAG = f'itr_{FOLD_IDX:03d}'
fold_seed = BASE_SEED + FOLD_IDX + 1
fold_dir = CHECKPOINT_ROOT / FOLD_TAG
if not fold_dir.is_dir():
    raise FileNotFoundError(f'未找到折目录: {fold_dir}')

print(f'=== {FOLD_TAG} | seed={fold_seed} ===')
model, fold_args = build_adsstnet_for_fold(fold_seed)
load_fold_checkpoint(model, fold_dir)
_, test_loader = load_test_loader(fold_args)

asm_by_layer, label_list, layer_ids = collect_test_freq_mask_gate(
    model,
    test_loader,
    correct_only=COLLECT_CORRECT_ONLY,
    balance_classes=BALANCE_CLASSES,
    balance_seed=fold_seed,
    class_ids=CLASS_IDS,
)

fold_out_dir = MASK_GATE_OUTPUT_DIR / FOLD_TAG
fold_out_dir.mkdir(parents=True, exist_ok=True)

freq_mask_stats_by_layer = {}
gate_stats_by_layer = {}
saved_files = []

for layer_idx in layer_ids:
    fm_stats, fm_counts = compute_freq_mask_count_by_class(
        asm_by_layer[layer_idx]['freq_mask'], label_list, CLASS_IDS,
    )
    gt_stats, gt_counts = compute_gate_mean_by_class(
        asm_by_layer[layer_idx]['gate'], label_list, CLASS_IDS,
    )
    freq_mask_stats_by_layer[layer_idx] = fm_stats
    gate_stats_by_layer[layer_idx] = gt_stats

    for c in CLASS_IDS:
        if c in fm_stats:
            print(
                f'Layer {layer_idx} | {CLASS_NAMES[c]} freq_mask: '
                f'n={fm_counts[c]}, shape={fm_stats[c].shape}, '
                f'max_count={fm_stats[c].max()}'
            )
        if c in gt_stats:
            print(
                f'Layer {layer_idx} | {CLASS_NAMES[c]} gate mean: '
                f'n={gt_counts[c]}, shape={gt_stats[c].shape}, '
                f'mean={gt_stats[c].mean():.4f}'
            )

    mask_pass_plot_path = fold_out_dir / f'{FOLD_TAG}_layer{layer_idx:02d}_freq_mask_pass_rate.svg'
    gate_plot_path = fold_out_dir / f'{FOLD_TAG}_layer{layer_idx:02d}_gate_mean.svg'
    plot_freq_mask_pass_rate_layer(
        fm_stats, fm_counts, layer_idx, FOLD_TAG, mask_pass_plot_path,
        input_seq_len=fold_args.seq_len, fs=SAMPLING_RATE,
    )
    plot_gate_mean_layer(
        gt_stats, gt_counts, layer_idx, FOLD_TAG, gate_plot_path,
        input_seq_len=fold_args.seq_len, fs=SAMPLING_RATE,
    )
    saved_files.extend([mask_pass_plot_path, gate_plot_path])
    print(
        f'  已保存 Layer {layer_idx}: '
        f'{mask_pass_plot_path.name}, {gate_plot_path.name}'
    )

# npz_path = fold_out_dir / f'{FOLD_TAG}_freq_mask_gate_stats.npz'
# npz_payload = {'layer_ids': np.array(layer_ids), 'labels': np.array(label_list)}
# for layer_idx in layer_ids:
#     for c in CLASS_IDS:
#         if c in freq_mask_stats_by_layer[layer_idx]:
#             npz_payload[f'layer{layer_idx}_class{c}_freq_mask_count'] = (
#                 freq_mask_stats_by_layer[layer_idx][c]
#             )
#         if c in gate_stats_by_layer[layer_idx]:
#             npz_payload[f'layer{layer_idx}_class{c}_gate_mean'] = (
#                 gate_stats_by_layer[layer_idx][c]
#             )
# np.savez(npz_path, **npz_payload)

=== itr_001 | seed=40 ===
train ids: [1, 2, 3, 4, 7, 9, 10, 11, 14, 16, 17, 18, 20, 23, 26, 27, 28, 29, 33, 34, 36, 37, 39, 40, 41, 42, 45, 46, 47, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 62, 64, 65, 66, 67, 68, 69, 70, 72, 73, 77, 79, 80, 81, 82, 83, 84, 85, 86, 90, 95, 96, 98, 99, 101, 103, 105, 107, 108, 110, 111] (数量: 70)
test ids: [5, 12, 13, 19, 22, 24, 32, 43, 44, 48, 59, 60, 61, 63, 71, 75, 78, 88, 92, 100, 102, 106, 109] (数量: 23)
collect_test_freq_mask_gate: 保留正确分类样本 2084/4080
collect_test_freq_mask_gate: 类别均衡采样 -> 每类 n=114 (HC: 1153->114, MCI: 817->114, AD: 114->114)
Layer 0 | HC freq_mask: n=114, shape=(20, 65), max_count=48
Layer 0 | HC gate mean: n=114, shape=(65,), mean=0.8089
Layer 0 | MCI freq_mask: n=114, shape=(20, 65), max_count=41
Layer 0 | MCI gate mean: n=114, shape=(65,), mean=0.8072
Layer 0 | AD freq_mask: n=114, shape=(20, 65), max_count=74
Layer 0 | AD gate mean: n=114, shape=(65,), mean=0.8041
  已保存 Layer 0: itr_001_layer00_freq_mask_pass_rate.svg, itr_001_la

## 绘制 ASM_before 和 ASM_after 的 PSD 图

In [22]:
from collections import defaultdict
from scipy.signal import welch



def _layer_fs_and_seq_len(layer_idx: int, input_seq_len: int, fs: float):
    seq_len = input_seq_len // (2 ** layer_idx)
    fs_eff = fs / (2 ** layer_idx)
    return seq_len, fs_eff


def compute_sample_psd_welch(signal_lc: np.ndarray, fs: float):
    """对单样本时序信号 (L, C) 用 Welch 法计算各通道 PSD，返回 freqs 与 (C, n_freqs)。"""
    signal_lc = np.asarray(signal_lc, dtype=np.float64)
    seq_len, n_ch = signal_lc.shape
    nperseg = min(seq_len, max(8, seq_len // 2)) if seq_len >= 4 else seq_len
    nperseg = max(2, int(nperseg))
    noverlap = nperseg // 2 if nperseg > 2 else 0

    freqs = None
    psd_list = []
    for ch in range(n_ch):
        f, pxx = welch(
            signal_lc[:, ch], fs=fs, nperseg=nperseg, noverlap=noverlap,
        )
        if freqs is None:
            freqs = f
        psd_list.append(pxx)
    return freqs, np.stack(psd_list, axis=0)


def compute_category_avg_psd_from_asm_samples(
    samples,
    labels,
    class_ids,
    fs: float,
):
    """逐样本计算 PSD，按人群聚合得到每通道平均 PSD。

    Returns
    -------
    freqs : np.ndarray
    category_avg_psd : dict[int, dict[int, np.ndarray]]
        {class_id: {channel_idx: avg_psd (n_freqs,)}}
    class_counts : dict[int, int]
    """
    labels = np.asarray(labels, dtype=int)
    class_ids = sorted(class_ids)

    sample_psds = []
    freqs = None
    for sig in samples:
        if sig is None:
            continue
        f, psd = compute_sample_psd_welch(sig, fs)
        freqs = f
        sample_psds.append(psd)

    if freqs is None:
        raise RuntimeError('无有效 ASM 样本，无法计算 PSD')

    category_data = {c: defaultdict(list) for c in class_ids}
    for idx, psd in enumerate(sample_psds):
        label = int(labels[idx])
        if label not in category_data:
            continue
        for ch_idx in range(psd.shape[0]):
            category_data[label][ch_idx].append(psd[ch_idx])

    category_avg_psd = {}
    class_counts = {}
    for c in class_ids:
        ch_dict = category_data[c]
        if not ch_dict:
            continue
        class_counts[c] = int(np.sum(labels == c))
        category_avg_psd[c] = {
            ch_idx: np.mean(np.stack(psd_list, axis=0), axis=0)
            for ch_idx, psd_list in ch_dict.items()
        }
    return freqs, category_avg_psd, class_counts


def plot_asm_psd_envelope_with_mean(
    category_avg_psd: dict,
    class_counts: dict,
    freqs: np.ndarray,
    layer_idx: int,
    asm_key: str,
    fold_tag: str,
    save_path: Path,
    class_ids=None,
    channels=None,
    fs: float = SAMPLING_RATE,
    input_seq_len: int = INPUT_SEQ_LEN,
    alpha: float = 0.35,
):
    """绘制 ASM 各人群通道 PSD 包络（min-max）及跨通道均值曲线。"""
    class_ids = class_ids or CLASS_IDS
    categories = [c for c in class_ids if c in category_avg_psd]
    if not categories:
        print(f'Warning: Layer {layer_idx} {asm_key} 无可用类别 PSD 数据')
        return

    if channels is None:
        n_ch = len(next(iter(category_avg_psd[categories[0]].values())))
        channels = list(range(n_ch))

    category_colors = {0: 'green', 1: 'orange', 2: 'red'}
    unit_label = 'μV²/Hz'

    all_avg_psd_values = []
    for c in categories:
        for ch_idx in channels:
            if ch_idx in category_avg_psd[c]:
                all_avg_psd_values.extend(category_avg_psd[c][ch_idx].flatten())
    if not all_avg_psd_values:
        print(f'Warning: Layer {layer_idx} {asm_key} 无 PSD 数据可绘制')
        return

    print(
        f'Layer {layer_idx} | {asm_key} PSD range: '
        f'min={np.min(all_avg_psd_values):.2e}, '
        f'max={np.max(all_avg_psd_values):.2e}, '
        f'mean={np.mean(all_avg_psd_values):.2e}'
    )

    fig, ax = _create_line_fig_ax()

    for c in categories:
        color = category_colors.get(c, 'black')
        cat_name = CLASS_NAMES[c] if c < len(CLASS_NAMES) else str(c)

        category_psd_data = [
            category_avg_psd[c][ch_idx]
            for ch_idx in channels
            if ch_idx in category_avg_psd[c]
        ]
        if not category_psd_data:
            continue

        category_psd_array = np.asarray(category_psd_data)
        min_psd = np.min(category_psd_array, axis=0)
        max_psd = np.max(category_psd_array, axis=0)
        mean_psd = np.mean(category_psd_array, axis=0)

        ax.fill_between(
            freqs, min_psd, max_psd,
            color=color, alpha=alpha,
            edgecolor=color, linewidth=1.0,
        )
        ax.plot(
            freqs, mean_psd,
            color=color, linewidth=LINE_WIDTH,
            label=f'{cat_name}',
        )

    _set_freq_xaxis(ax, freqs, layer_idx=layer_idx)
    _set_line_ylabel(ax, f'PSD ({unit_label})')

    _style_line_fig_ax(ax)
    _style_line_yticks(ax)
    _style_line_legend(ax)
    # _set_line_title(
    #     ax,
    #     f'{fold_tag} | Layer {layer_idx} | {asm_key} seq_len={seq_len_layer}, fs={fs_eff:.0f} Hz',
    # )
    _save_line_fig(fig, save_path)


def plot_asm_psd_mean_only(
    category_avg_psd: dict,
    class_counts: dict,
    freqs: np.ndarray,
    layer_idx: int,
    asm_key: str,
    fold_tag: str,
    save_path: Path,
    class_ids=None,
    channels=None,
    fs: float = SAMPLING_RATE,
    input_seq_len: int = INPUT_SEQ_LEN,
):
    """绘制 ASM 各人群跨通道平均 PSD 均值曲线（不含包络）。"""
    class_ids = class_ids or CLASS_IDS
    categories = [c for c in class_ids if c in category_avg_psd]
    if not categories:
        print(f'Warning: Layer {layer_idx} {asm_key} 无可用类别 PSD 数据')
        return

    if channels is None:
        n_ch = len(next(iter(category_avg_psd[categories[0]].values())))
        channels = list(range(n_ch))

    category_colors = {0: 'green', 1: 'orange', 2: 'red'}
    unit_label = 'μV²/Hz'

    fig, ax = _create_line_fig_ax()

    for c in categories:
        color = category_colors.get(c, 'black')
        cat_name = CLASS_NAMES[c] if c < len(CLASS_NAMES) else str(c)

        category_psd_data = [
            category_avg_psd[c][ch_idx]
            for ch_idx in channels
            if ch_idx in category_avg_psd[c]
        ]
        if not category_psd_data:
            continue

        mean_psd = np.mean(np.asarray(category_psd_data), axis=0)
        ax.plot(
            freqs, mean_psd,
            color=color, linewidth=LINE_WIDTH,
            label=f'{cat_name}',
        )

    _set_freq_xaxis(ax, freqs, layer_idx=layer_idx)
    _set_line_ylabel(ax, f'PSD ({unit_label})')

    _style_line_fig_ax(ax)
    _style_line_yticks(ax)
    _style_line_legend(ax)
    # _set_line_title(
    #     ax,
    #     f'{fold_tag} | Layer {layer_idx} | {asm_key} mean PSD seq_len={seq_len_layer}, fs={fs_eff:.0f} Hz',
    # )
    _save_line_fig(fig, save_path)


# ========== 对每层 ASM_before / ASM_after 计算 PSD 并绘图 ==========
fold_out_dir.mkdir(parents=True, exist_ok=True)

asm_psd_keys = ('ASM_before', 'ASM_after')
asm_psd_saved = []

for layer_idx in layer_ids:
    _, fs_eff = _layer_fs_and_seq_len(layer_idx, fold_args.seq_len, SAMPLING_RATE)

    for asm_key in asm_psd_keys:
        samples = asm_by_layer[layer_idx][asm_key]
        freqs, cat_avg_psd, cat_counts = compute_category_avg_psd_from_asm_samples(
            samples, label_list, CLASS_IDS, fs=fs_eff,
        )
        # out_path = asm_psd_out_dir / (
        #     f'{FOLD_TAG}_layer{layer_idx:02d}_{asm_key}_psd_envelope.png'
        # )
        # plot_asm_psd_envelope_with_mean(
        #     cat_avg_psd, cat_counts, freqs,
        #     layer_idx, asm_key, FOLD_TAG, out_path,
        #     class_ids=CLASS_IDS,
        #     fs=SAMPLING_RATE,
        #     input_seq_len=fold_args.seq_len,
        # )
        # asm_psd_saved.append(out_path)
        # print(f'  已保存: {out_path.name}')

        mean_path = fold_out_dir / (
            f'{FOLD_TAG}_layer{layer_idx:02d}_{asm_key}_psd_mean.svg'
        )
        plot_asm_psd_mean_only(
            cat_avg_psd, cat_counts, freqs,
            layer_idx, asm_key, FOLD_TAG, mean_path,
            class_ids=CLASS_IDS,
            fs=SAMPLING_RATE,
            input_seq_len=fold_args.seq_len,
        )
        asm_psd_saved.append(mean_path)
        print(f'  已保存: {mean_path.name}')

print(f'\n共保存 {len(asm_psd_saved)} 张 ASM PSD 图 -> {fold_out_dir}')


  已保存: itr_001_layer00_ASM_before_psd_mean.svg
  已保存: itr_001_layer00_ASM_after_psd_mean.svg
  已保存: itr_001_layer01_ASM_before_psd_mean.svg
  已保存: itr_001_layer01_ASM_after_psd_mean.svg
  已保存: itr_001_layer02_ASM_before_psd_mean.svg
  已保存: itr_001_layer02_ASM_after_psd_mean.svg
  已保存: itr_001_layer03_ASM_before_psd_mean.svg
  已保存: itr_001_layer03_ASM_after_psd_mean.svg
  已保存: itr_001_layer04_ASM_before_psd_mean.svg
  已保存: itr_001_layer04_ASM_after_psd_mean.svg
  已保存: itr_001_layer05_ASM_before_psd_mean.svg
  已保存: itr_001_layer05_ASM_after_psd_mean.svg

共保存 12 张 ASM PSD 图 -> /root/autodl-tmp/ADSST-Net/figs/Adaptive Spectral Modulation/itr_001
